# Tensor 

## Custom architecture of code for transformer of 200 million parameters

In [ ]:
import torch
import torch.nn as nn


### Config

In [2]:
# Embedding dimension of transformer model
embedding_dim = 512
# feed forward network 
input_size = embedding_dim
hidden_size = embedding_dim * 4
output_size = embedding_dim

### Feed forward network  

In [3]:
# feedforward function for the custom model
class FeedForward(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(FeedForward, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.selu = torch.nn.SiLU()
        self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
        self.selu = torch.nn.SiLU()
        self.fc3 = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x_1= self.fc1(x)
        out = self.selu(x_1)
        x_2 = self.fc2(out)
        out = self.selu(x_2)
        out = self.fc3(out)
        return out

# Layer Normolization 

### RMSNorm (Root Mean Square Normalization)

RMSNorm is a simpler version of LayerNorm. It only rescales each token's vector by its root mean square. There is no mean subtraction and no bias, so it is faster and has fewer parameters.

$$\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

$$\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot g_i$$

where:
- $d$ = embedding dimension
- $g$ = learnable weight (one per feature)
- $\epsilon$ = small constant to avoid division by zero

Used in LLaMA, Mistral, Gemma and most modern LLMs.

In [ ]:
# using RMSNorm (Root Mean Square Layer Normalization) 
class layer_norm(nn.Module):
    def __init__(self,embedding_dim,eps = 1e-5):
        super(layer_norm,self).__init__()
        self.eps = eps
        self.embedding_dim = embedding_dim
        self.weight = nn.Parameter(torch.ones(embedding_dim))
        
    def forward(self,x):
        rms_norm = nn.RMSNorm(self.embedding_dim, eps=self.eps)
        return rms_norm(x) * self.weight
        